# Modeling: Personality Type Predictor

This notebook loads `data/data_processed.csv`, separates a stratified holdout, and evaluates complete preprocessing-and-classifier pipelines. It compares a dummy baseline and three model families, tunes each family using training-only cross-validation, records trials in MLflow, and reports the selected champion's holdout performance.

Model selection uses mean macro-F1 on training folds, with class-wise performance, stability, runtime, and explainability as supporting considerations. The holdout is not used to choose the model. Run the notebook from the project root after generating the cleaned dataset.

## Workflow and evaluation safeguards

`X` contains the 19 questionnaire items plus `age`, `gender`, and `hand`; `y` is the four-class `target`. A stratified 80/20 split preserves class proportions. Within each training fold, the complete `Pipeline` fits imputation, scaling, encoding, and the classifier afresh, preventing preprocessing leakage.

The dummy classifier is a minimum reference. Logistic regression, KNN, and random forest are first compared with stratified five-fold CV and then tuned with bounded Hyperopt searches. Mean macro-F1 is the primary metric; standard deviation, fit time, and out-of-fold class-wise F1 support interpretation. MLflow stores each trial's parameters, CV metrics, and fitted pipeline. The final cell retrieves only the new run ID of the already selected champion configuration and evaluates it on the holdout.

In [1]:
# Path represents file and directory paths across operating systems.
from pathlib import Path

# Pandas stores tabular data in DataFrames; NumPy provides numerical utilities.
import numpy as np
import pandas as pd

# MLflow records experiments, parameters, metrics, and fitted model pipelines.
import mlflow
import mlflow.sklearn

# Hyperopt searches predefined parameter spaces.
# Trials collects attempts; fmin starts the search; tpe.suggest proposes the next trial.
# hp defines search spaces; scope.int converts sampled whole numbers to Python integers.
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from hyperopt.pyll import scope

# These scikit-learn components separate and process numerical and categorical features.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# The four classifier classes cover naive, linear, distance-based, and tree-based learning.
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

# Metrics evaluate predictions; the report and confusion matrix show class-wise results.
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score

# These functions split data and perform repeated leakage-safe validation.
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, cross_validate, train_test_split

# Key constants are grouped here so paths, the target, and random choices are easy to find.
DATA_PATH = Path("data/data_processed.csv")
TARGET_COLUMN = "target"
DEMOGRAPHIC_COLUMNS = ["age", "gender", "hand"]

# Reserve 20% as the holdout; seed 42 fixes randomized steps.
TEST_SIZE = 0.20
RANDOM_STATE = 42


## 1. Load cleaned data and check feature roles

In [2]:
# pd.read_csv loads the cleaned CSV into a DataFrame.
df = pd.read_csv(DATA_PATH)

# drop() removes only the specified column, creating X with all inputs.
# y is a pandas Series containing each row's class to predict.
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

# Feature groups follow the agreed column roles.
# The list comprehension iterates over X columns and keeps non-demographic columns.
question_columns = [
    column
    for column in X.columns
    if column not in DEMOGRAPHIC_COLUMNS
]
# + creates a new list: the 19 response items and age are numerical.
numeric_features = question_columns + ["age"]
categorical_features = ["gender", "hand"]

# assert checks expectations and stops execution with a message if they fail.
# It changes neither data nor model parameters.
assert len(question_columns) == 19, "Expected 19 questionnaire features."
assert X.shape[1] == 22, "Expected 22 input features."
assert y.notna().all(), "The target column must not have missing values."

print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Numerical features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

Dataset: 19715 rows, 23 columns
Numerical features (20): ['N6', 'N8', 'N7', 'N1', 'N9', 'N10', 'N3', 'N5', 'E3', 'N2', 'E5', 'N4', 'E7', 'E4', 'C4', 'E1', 'A4', 'E10', 'E9', 'age']
Categorical features (2): ['gender', 'hand']


## 2. Stratified train/test split

We hold out 20% of observations for the final test. `stratify=y` preserves the proportions of the four target classes in both partitions. `random_state=42` makes the split reproducible.

In [3]:
# train_test_split selects whole rows with a reproducible random procedure.
# X and y are passed together so each input remains paired with its class.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,  # Reserves 20% of rows for final generalization assessment.
    random_state=RANDOM_STATE,  # The same seed reproduces the split.
    stratify=y,  # Preserves approximately equal target-class proportions in both partitions.
)

print(f"Training: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Test:     X_test={X_test.shape}, y_test={y_test.shape}")

Training: X_train=(15772, 22), y_train=(15772,)
Test:     X_test=(3943, 22), y_test=(3943,)


In [4]:
# value_counts(normalize=True) returns proportions rather than counts.
# The dictionary assigns column labels; concat(..., axis=1) places the three series side by side.
split_class_distribution = pd.concat(
    {
        "Overall": y.value_counts(normalize=True),
        "Training": y_train.value_counts(normalize=True),
        "Test": y_test.value_counts(normalize=True),
    },
    axis=1,
)
# Replace dictionary keys with presentation-ready column labels.
split_class_distribution.columns = ["Overall (%)", "Training (%)", "Test (%)"]
# Multiplying by 100 converts proportions to percentages; round(2) affects display only.
(split_class_distribution * 100).round(2)

,Overall (%),Training (%),Test (%)
target,,,
Moderate,42.86,42.85,42.86
Resilient,31.26,31.26,31.25
Overcontroller,14.35,14.35,14.35
Undercontroller,11.54,11.54,11.54


## 3. Preprocessing pipeline: agreed approach

The 19 questionnaire items and `age` are processed as numerical features. Median imputation remains important for `age` because 83 age values are missing after the documented cleaning. The median is more robust to unusually high or low values than the mean. The questionnaire items have no missing values after cleaning, but they pass through the same numerical pipeline for consistent handling.

`gender` and `hand` are nominal categories, not ordered scales. One-hot encoding creates a separate 0/1 column for each category observed during training instead of implying an artificial order such as `0 < 1 < 2`. `handle_unknown="ignore"` prevents an unseen category from crashing the pipeline. The Streamlit app should nevertheless offer only known choices; this setting is an additional robustness measure.

To prevent data leakage, all subsequent parameters are learned from `X_train` only. `X_test` is neither scaled nor transformed here and remains reserved for champion evaluation.

In [5]:
# SimpleImputer learns medians from X_train only and fills missing numerical values.
# StandardScaler centers numerical columns at zero and scales them to unit standard deviation.
# steps is an ordered list of (name, transformer) pairs; names aid inspection.
numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),  # A median is learned for each numerical column.
        ("scaler", StandardScaler()),  # Questionnaire items and age receive comparable scales.
    ]
)

# The most frequent category fills missing gender and hand values.
# OneHotEncoder creates an indicator column per category.
# For example, gender_Female can be 0 or 1 independently of gender_Male.
# handle_unknown="ignore" avoids an error for a later unseen category.
categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),  # Most frequent value per categorical column.
        ("encoder", OneHotEncoder(handle_unknown="ignore")),  # An unseen category produces zeros instead of an error.
    ]
)

# ColumnTransformer applies each sub-pipeline only to its assigned columns
# and combines their outputs into one feature matrix.
# remainder="drop" is explicit: only the 22 agreed inputs proceed.
# verbose_feature_names_out=False keeps output names short, e.g. gender_Female instead of categorical__gender_Female.
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessor, numeric_features),
        ("categorical", categorical_preprocessor, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

## 4. Inspect preprocessing on training data only

`fit_transform(X_train)` learns imputation values, scaling parameters, and category columns exclusively from the training set and then transforms it. The checks verify only technical properties: unchanged row count, traceable feature names, and no remaining missing values. No classifier is trained and no test result is inspected at this stage.

In [6]:
# fit_transform() learns all parameters from X_train only and creates the training matrix.
X_train_preprocessed = preprocessor.fit_transform(X_train)
transformed_feature_names = preprocessor.get_feature_names_out()

# These assertions check the transformation's technical integrity.
assert X_train_preprocessed.shape[0] == X_train.shape[0], "The row count must not change."
assert X_train_preprocessed.shape[1] == len(transformed_feature_names), "Feature names and matrix columns must match."
assert not pd.isna(X_train_preprocessed).any(), "No missing values may remain after preprocessing."

preprocessing_overview = pd.DataFrame(
    {
        "Metric": ["Input rows (training)", "Raw features", "Features after preprocessing", "Missing values after preprocessing"],
        "Value": [
            X_train.shape[0],
            X_train.shape[1],
            X_train_preprocessed.shape[1],
            int(pd.isna(X_train_preprocessed).sum()),
        ],
    }
)

display(preprocessing_overview)
print("Features after preprocessing:")
print(list(transformed_feature_names))

,Metric,Value
0,Input rows (training),15772
1,Raw features,22
2,Features after preprocessing,26
3,Missing values after preprocessing,0


Features after preprocessing:
['N6', 'N8', 'N7', 'N1', 'N9', 'N10', 'N3', 'N5', 'E3', 'N2', 'E5', 'N4', 'E7', 'E4', 'C4', 'E1', 'A4', 'E10', 'E9', 'age', 'gender_Female', 'gender_Male', 'gender_Other', 'hand_Both', 'hand_Left', 'hand_Right']


## 5. Dummy baseline with stratified 5-fold cross-validation

The dummy baseline always predicts the most frequent training class, `Moderate`. It learns no relationships from the inputs and is not a serious model candidate. It sets a minimum reference: a useful classifier should clearly outperform this strategy on `f1_macro`.

`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` creates five reproducible train/validation splits with similar class proportions. In each iteration, the complete pipeline is fitted from scratch on four folds and evaluated on the fifth. Imputation, scaling, and encoding are thus learned from each fold's training partition only.

We summarize the five scores by their mean and sample standard deviation. The holdout remains untouched by this step.

In [7]:
# StratifiedKFold approximately preserves class proportions in each fold.
# shuffle=True and random_state make the exact splits reproducible.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# The Pipeline joins preprocessing and baseline so each CV fold fits both together.
# most_frequent always predicts the current training fold's most common class.
dummy_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent")),
    ]
)

# cross_val_score clones the whole pipeline for each fold.
# No learned imputation, scaling, or encoding parameters are shared across folds.
dummy_cv_scores = cross_val_score(
    dummy_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_macro",
)

dummy_cv_summary = pd.DataFrame(
    {
        "Metric": [
            "Fold 1 macro-F1",
            "Fold 2 macro-F1",
            "Fold 3 macro-F1",
            "Fold 4 macro-F1",
            "Fold 5 macro-F1",
            "Mean macro-F1",
            "Macro-F1 SD",
        ],
        "Value": [*dummy_cv_scores, dummy_cv_scores.mean(), dummy_cv_scores.std(ddof=1)],
    }
)

display(dummy_cv_summary.round(3))

,Metric,Value
0,Fold 1 macro-F1,0.15
1,Fold 2 macro-F1,0.15
2,Fold 3 macro-F1,0.15
3,Fold 4 macro-F1,0.15
4,Fold 5 macro-F1,0.15
5,Mean macro-F1,0.15
6,Macro-F1 SD,0.00


### Interpreting the baseline

A low macro-F1 is expected: the dummy classifier identifies some `Moderate` cases but never identifies the other three classes. This is not evidence of overfitting; it is a deliberately underfitting reference. The real candidates use the same cross-validation logic for a comparable assessment.

## 6. Compare initial model candidates

We compare three model families: logistic regression as a linear, explainable model; KNN as a distance-based model; and random forest as a tree ensemble that can capture nonlinear patterns and interactions.

This is not hyperparameter tuning. Candidates start near library defaults. `max_iter=1000` for logistic regression allows enough optimization iterations for convergence; it is not itself a performance-tuning search. `random_state=42` makes randomized steps reproducible.

Each classifier is evaluated with a fresh clone of the same preprocessor in the same stratified 5-fold cross-validation. Mean macro-F1 is the primary comparison metric; variation and fit time add context.

In [8]:
# All candidates start from a documented baseline configuration.
# KNN uses default k=5; random forest uses its default 100 trees.
# These values are revisited during tuning.
# The dictionary maps each label to an unfitted classifier instance.
model_candidates = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Random Forest (100 trees)": RandomForestClassifier(random_state=RANDOM_STATE),
}

comparison_rows = []

# items() yields a result-table label and classifier in each iteration.
for model_name, model in model_candidates.items():
    # Each pipeline contains preprocessing and a classifier.
    # cross_validate() clones the pipeline per fold, preventing preprocessing leakage.
    candidate_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    # cross_validate returns test_score and fit_time per fold for comparison.
    cv_result = cross_validate(
        candidate_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="f1_macro",
        return_train_score=False,
    )

    # append() adds a dictionary of summary values to the result list.
    comparison_rows.append(
        {
            "Model": model_name,
            "CV macro-F1 (mean)": cv_result["test_score"].mean(),
            "CV macro-F1 (SD)": cv_result["test_score"].std(ddof=1),
            "Mean fit time (s)": cv_result["fit_time"].mean(),
        }
    )

# Convert the list into a table and sort descending by the primary metric.
# reset_index(drop=True) gives the sorted table a clean index.
model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("CV macro-F1 (mean)", ascending=False)
    .reset_index(drop=True)
)

assert model_comparison["CV macro-F1 (mean)"].between(0, 1).all(), "F1 values must be between 0 and 1."
display(model_comparison.round(3))

,Model,CV macro-F1 (mean),CV macro-F1 (SD),Mean fit time (s)
0,Logistic Regression,0.768,0.006,0.265
1,Random Forest (100 trees),0.744,0.013,1.048
2,KNN (k=5),0.722,0.007,0.052


### Initial comparison

Before tuning, logistic regression reached mean CV macro-F1 of 0.768 (SD 0.006), random forest 0.744 (SD 0.013), and KNN 0.722. The dummy baseline reached 0.150. These are historical results from the documented project run; regenerated outputs should be checked against them.

The ranking does not itself select a champion. All three families proceed to bounded tuning, followed by a training-only comparison. Holdout performance is reported only after model selection.

## 7. Tuning scope and MLflow tracking

The Learning Journey calls for documented hyperparameter tuning, comparison of multiple classifiers using macro-F1, and MLflow tracking. It does not prescribe exact trial counts or search ranges. Anuar's later written project description asks for tuning each of the three compared families. This notebook therefore uses 20 Hyperopt trials per family: `C` and `class_weight` for logistic regression; tree structure, feature sampling, and class weight for random forest; and neighbor count, vote weighting, and distance metric for KNN.

Each trial evaluates the full pipeline using stratified five-fold CV on `X_train` and `y_train`. The logged model is then fitted on all training rows, while its CV metric remains the selection evidence. No holdout result is used during tuning.

In [9]:
# Derive the tracking store from the project root at runtime.
# This lets the notebook, MLflow UI, and app use the same local store
# without hard-coding a user-specific absolute path.
# cwd() is the current project directory; resolve() makes its path absolute.
PROJECT_ROOT = Path.cwd().resolve()
# The / operator joins Path objects into a file path.
MLFLOW_DB_PATH = PROJECT_ROOT / "mlflow.db"
# as_posix() uses forward slashes suitable for SQLite URIs on Windows.
MLFLOW_TRACKING_URI = f"sqlite:///{MLFLOW_DB_PATH.as_posix()}"
MLFLOW_EXPERIMENT_NAME = "personality-type-predictor"
TUNING_TRIALS_PER_MODEL = 20
# MLflow 3 stores scikit-learn models in the safer skops format by default.
# Both trusted types come only from pipelines trained in this notebook.
SKOPS_TRUSTED_TYPES = ["numpy.dtype", "sklearn.tree._tree.Tree"]

# These settings apply to subsequent mlflow.start_run() calls in this kernel.
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

# Report the relative store location, not a private absolute machine path.
print("MLflow store: ./mlflow.db (SQLite; resolved at runtime)")
print(f"Experiment: {MLFLOW_EXPERIMENT_NAME}")
print(f"Hyperopt trials per model: {TUNING_TRIALS_PER_MODEL}")

MLflow store: ./mlflow.db (SQLite; resolved at runtime)
Experiment: personality-type-predictor
Hyperopt trials per model: 20


## 8. Hyperopt search spaces

Hyperopt searches only within the predefined spaces below. `C` uses a logarithmic scale because a change from 0.01 to 0.1 may matter as much as a change from 1 to 10. The discrete, bounded random-forest choices are easier to explain than an unrestricted space.

The random-forest `max_depth` choices include 3, 5, 10, 20, and `None`, as demonstrated in Live Session 2. Additional parameters are our justified extension, not a mandatory Learning Journey requirement. For KNN, `n_neighbors` controls how strongly individual cases affect predictions: smaller values are more flexible but may react to noise; larger values smooth more and may miss local patterns. `weights="uniform"` counts all neighbors equally, while `weights="distance"` gives nearby neighbors more influence. `p=1` uses Manhattan distance (sum of absolute differences), and `p=2` uses Euclidean distance (straight-line distance). Scaling remains inside the CV pipeline and is re-fitted per fold.

In [10]:
# hp.loguniform() samples C logarithmically between 0.001 and 100.
# hp.choice() selects among defined categorical options.
logistic_search_space = {
    "C": hp.loguniform("logreg_C", np.log(0.001), np.log(100.0)),  # Continuous regularization parameter.
    "class_weight": hp.choice("logreg_class_weight", [None, "balanced"]),  # Without or with class weighting.
}

# scope.int() converts numbers from quniform() into integer model parameters.
random_forest_search_space = {
    "n_estimators": scope.int(hp.quniform("rf_n_estimators", 200, 500, 50)),  # 200, 250, ..., 500 trees.
    "max_depth": hp.choice("rf_max_depth", [3, 5, 10, 20, None]),  # Limited depth or no explicit depth limit.
    "min_samples_split": scope.int(hp.quniform("rf_min_samples_split", 2, 12, 1)),  # Integers from 2 to 12.
    "min_samples_leaf": scope.int(hp.quniform("rf_min_samples_leaf", 1, 6, 1)),  # Integers from 1 to 6.
    "max_features": hp.choice("rf_max_features", ["sqrt", "log2", 0.5]),  # Three explainable feature-selection choices.
    "class_weight": hp.choice("rf_class_weight", [None, "balanced"]),  # Without or with class weighting.
}

# n_neighbors controls smoothing; odd values reduce some voting ties.
# With 20 trials, Hyperopt tests only some possible combinations.
knn_search_space = {
    "n_neighbors": hp.choice("knn_n_neighbors", [3, 5, 7, 11, 15, 21, 31, 51]),
    "weights": hp.choice("knn_weights", ["uniform", "distance"]),
    "p": hp.choice("knn_p", [1, 2]),
}

print("Logistic Regression: C, class_weight")
print("Random Forest: n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features, class_weight")
print("KNN: n_neighbors, weights, p")

Logistic Regression: C, class_weight
Random Forest: n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features, class_weight
KNN: n_neighbors, weights, p


## 9. Reusable trial function

The same trial procedure serves all three families: construct the complete pipeline, measure CV performance, and log parameters, mean, variation, fit time, and the pipeline fitted on `X_train`. `loss` is negative mean macro-F1 because Hyperopt minimizes its objective.

A logged model artifact is fitted on all training rows, but its logged CV metric is the basis for selection. The holdout had already been evaluated for the existing champion and must not be used to revise model choice after the KNN addendum.

In [11]:
def normalise_params(params):
    """Convert Hyperopt/NumPy values to Python types suitable for scikit-learn and MLflow.

    Hyperopt sometimes returns NumPy-specific number types. This function
    converts them to built-in Python numbers so parameters can be logged clearly
    and processed consistently by scikit-learn.
    """
    normalised = {}
    # items() yields each parameter name and its sampled value.
    for name, value in params.items():
        if isinstance(value, np.integer):
            normalised[name] = int(value)
        elif isinstance(value, np.floating):
            normalised[name] = float(value)
        else:
            normalised[name] = value
    return normalised


def build_tuning_model(model_name, params):
    """Build one unfitted model from the agreed search parameters.

    The function keeps model-family-specific parameters in one
    place. It does not fit the model; fit() happens in the pipeline.
    """
    if model_name == "Logistic Regression":
        return LogisticRegression(
            C=params["C"],  # Inverse regularization strength: larger C allows a more flexible fit.
            class_weight=params["class_weight"],  # balanced gives rarer training classes more weight.
            max_iter=1000,  # Iteration limit allows the optimizer to converge.
            random_state=RANDOM_STATE,  # Makes randomized solver steps reproducible.
        )

    if model_name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=params["n_estimators"],  # Number of trees in the ensemble.
            max_depth=params["max_depth"],  # None leaves depth to the other stopping rules.
            min_samples_split=params["min_samples_split"],  # Minimum rows required to split a node.
            min_samples_leaf=params["min_samples_leaf"],  # Minimum rows per leaf; limits overly specific rules.
            max_features=params["max_features"],  # Features considered at each split.
            class_weight=params["class_weight"],  # Optional weighting for unequal class sizes.
            random_state=RANDOM_STATE,  # Fixes randomized tree and feature selection.
            n_jobs=-1,  # Uses all available local CPU cores without changing model logic.
        )

    if model_name == "KNN":
        return KNeighborsClassifier(
            n_neighbors=params["n_neighbors"],  # Number of training neighbors considered.
            weights=params["weights"],  # Equal or distance-weighted votes.
            p=params["p"],  # 1: Manhattan distance; 2: Euclidean distance.
        )

    raise ValueError(f"Unknown model: {model_name}")


def run_hyperopt_search(model_name, search_space, max_evals):
    """Run a bounded Hyperopt search and return a trial table.

    Each trial builds a new pipeline and evaluates it by CV on
    training data, then stores a pipeline fitted on all training rows
    with its metrics in MLflow.
    """
    trial_records = []
    trials = Trials()

    def objective(raw_params):
        """Evaluate a parameter combination proposed by Hyperopt."""
        params = normalise_params(raw_params)
        model = build_tuning_model(model_name, params)
        # The complete pipeline fits preprocessing separately in each CV fold.
        pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", model),
            ]
        )

        # cross_validate clones the pipeline per fold; each fold fits its own preprocessing.
        cv_result = cross_validate(
            pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring="f1_macro",
            return_train_score=False,
        )
        # test_score contains five validation-fold scores, not the reserved holdout score.
        mean_score = float(cv_result["test_score"].mean())
        # ddof=1 computes sample standard deviation across folds.
        score_std = float(cv_result["test_score"].std(ddof=1))
        mean_fit_time = float(cv_result["fit_time"].mean())

        # After CV, fit the artifact on all training data; X_test stays excluded.
        fitted_pipeline = pipeline.fit(X_train, y_train)
        trial_number = len(trial_records) + 1
        # MLflow stores parameters as text; None is represented as a string in the table.
        mlflow_params = {name: ("None" if value is None else value) for name, value in params.items()}

        # The context manager opens and closes an MLflow run.
        with mlflow.start_run(run_name=f"{model_name}-trial-{trial_number:02d}") as active_run:
            mlflow.set_tags(
                {
                    "stage": "hyperparameter_tuning",
                    "model_family": model_name,
                    "model_serialization": "skops",
                    "skops_trusted_types": ",".join(SKOPS_TRUSTED_TYPES),
                    "selection_metric": "f1_macro",
                    "data_partition": "training_only",
                }
            )
            # ** expands the dictionary so the model family and sampled parameters are logged together.
            mlflow.log_params({"model_name": model_name, **mlflow_params})
            mlflow.log_metrics(
                {
                    "cv_f1_macro_mean": mean_score,
                    "cv_f1_macro_std": score_std,
                    "cv_fit_time_mean_seconds": mean_fit_time,
                }
            )
            # MLflow uses the safer skops format here. This notebook creates the pipeline;
            # only the two known scikit-learn/NumPy types reported by MLflow
            # are trusted for this artifact.
            mlflow.sklearn.log_model(
                fitted_pipeline,
                name="model",
                skops_trusted_types=SKOPS_TRUSTED_TYPES,
            )
            run_id = active_run.info.run_id

        trial_records.append(
            {
                "Model": model_name,
                "Trial": trial_number,
                "CV macro-F1 (mean)": mean_score,
                "CV macro-F1 (SD)": score_std,
                "Mean fit time (s)": mean_fit_time,
                "MLflow run ID": run_id,
                **params,
            }
        )

        # Hyperopt minimizes loss; negating macro-F1 makes its minimum the maximum score.
        return {"loss": -mean_score, "status": STATUS_OK}

    # fmin runs max_evals trials; the seeded NumPy generator reproduces the search sequence.
    fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=max_evals,
        trials=trials,
        rstate=np.random.default_rng(RANDOM_STATE),
        show_progressbar=False,
    )

    return pd.DataFrame(trial_records)


## 10. Run tuning and compare configurations

The three bounded searches run in sequence. The displayed table identifies the highest-CV-macro-F1 trial in each family; MLflow retains the full set of trials, parameters, metrics, and fitted pipelines. A family's best trial is a candidate for the subsequent champion review, not an automatic final selection.

In [12]:
# The function returns a row per trial; the three searches remain distinguishable.
logistic_tuning_results = run_hyperopt_search(
    model_name="Logistic Regression",
    search_space=logistic_search_space,
    max_evals=TUNING_TRIALS_PER_MODEL,
)

random_forest_tuning_results = run_hyperopt_search(
    model_name="Random Forest",
    search_space=random_forest_search_space,
    max_evals=TUNING_TRIALS_PER_MODEL,
)

# The KNN addendum uses the same training folds and primary metric.
knn_tuning_results = run_hyperopt_search(
    model_name="KNN",
    search_space=knn_search_space,
    max_evals=TUNING_TRIALS_PER_MODEL,
)

# concat(..., ignore_index=True) stacks and reindexes the three trial tables.
tuning_results = pd.concat(
    [logistic_tuning_results, random_forest_tuning_results, knn_tuning_results],
    ignore_index=True,
).sort_values("CV macro-F1 (mean)", ascending=False)

# groupby("Model") groups by family; idxmax() identifies the highest macro-F1 row.
# loc selects those three rows for manual comparison.
best_tuning_runs = (
    tuning_results.loc[
        tuning_results.groupby("Model")["CV macro-F1 (mean)"].idxmax()
    ]
    .sort_values("CV macro-F1 (mean)", ascending=False)
    .reset_index(drop=True)
)

# These checks verify trial completeness and the valid F1 range.
assert len(tuning_results) == 3 * TUNING_TRIALS_PER_MODEL, "Each trial must be recorded exactly once."
assert best_tuning_runs["CV macro-F1 (mean)"].between(0, 1).all(), "F1 values must be between 0 and 1."

display(best_tuning_runs.round(3))

,Model,Trial,CV macro-F1 (mean),CV macro-F1 (SD),Mean fit time (s),MLflow run ID,C,class_weight,max_depth,max_features,min_samples_leaf,min_samples_split,n_estimators,n_neighbors,p,weights
0,Random Forest,11,0.786,0.009,0.476,0beee6fd95aa49b79915be4854286137,NaN,balanced,NaN,log2,2.0,2.0,200.0,NaN,NaN,NaN
1,Logistic Regression,12,0.784,0.007,0.298,7d27288ce33e41d3b5490f8b04e9b07d,1.098,balanced,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,KNN,6,0.744,0.008,0.046,85c6a69fac5e4b93bc004cb4bbb74f52,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,2.0,distance


## Tie-breaker: class-wise CV performance

The best configurations are first compared by mean CV macro-F1 on `X_train`. For a justified manual champion decision, we also inspect out-of-fold predictions for all three families on `X_train` and `y_train`. Every row is predicted once by a pipeline that did not train on its fold. This is neither a new primary metric nor a holdout evaluation: macro-F1 remains primary, and class-wise results help interpret a close comparison.

In [13]:
def best_params_from_result(row):
    """Read one family's best configuration from the trial table.

    The table contains columns for three model families. Inapplicable
    columns are empty; the branch builds only the parameters
    accepted by the selected model.
    """
    if row["Model"] == "Logistic Regression":
        return {
            "C": float(row["C"]),
            "class_weight": row["class_weight"],
        }

    if row["Model"] == "KNN":
        return {
            "n_neighbors": int(row["n_neighbors"]),  # Convert the Hyperopt number to an integer for scikit-learn.
            "weights": row["weights"],
            "p": int(row["p"]),  # 1 or 2 for the selected distance metric.
        }

    if row["Model"] != "Random Forest":
        raise ValueError(f"Unknown model in trial table: {row['Model']}")

    # This return value is for random forest; int() restores integer hyperparameters.
    return {
        "n_estimators": int(row["n_estimators"]),
        # Pandas represents None as NaN in this mixed table.
        "max_depth": None if pd.isna(row["max_depth"]) else int(row["max_depth"]),
        "min_samples_split": int(row["min_samples_split"]),
        "min_samples_leaf": int(row["min_samples_leaf"]),
        "max_features": row["max_features"],
        "class_weight": row["class_weight"],
    }


# sorted() gives both reports a stable alphabetical class order.
class_labels = sorted(y_train.unique())
# This list collects one row per model and target class.
classwise_records = []

# iterrows() loops over three best rows; _ discards the unused index.
for _, best_row in best_tuning_runs.iterrows():
    model_name = best_row["Model"]
    # Rebuild the best configuration unfitted in the same preprocessing pipeline.
    diagnostic_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", build_tuning_model(model_name, best_params_from_result(best_row))),
        ]
    )

    # cross_val_predict makes a validation-fold prediction for each training row.
    oof_predictions = cross_val_predict(diagnostic_pipeline, X_train, y_train, cv=cv)
    # output_dict=True returns a dictionary of class-wise metrics for tabulation.
    # zero_division=0 returns zero for undefined ratios instead of warning.
    report = classification_report(
        y_train,
        oof_predictions,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )

    # Extract one table row per class from the report.
    for class_label in class_labels:
        classwise_records.append(
            {
                "Model": model_name,
                "Target class": class_label,
                "Precision": report[class_label]["precision"],
                "Recall": report[class_label]["recall"],
                "F1": report[class_label]["f1-score"],
                "Support": int(report[class_label]["support"]),
            }
        )


# The final DataFrame compares class-wise results across all three models.
classwise_cv_results = pd.DataFrame(classwise_records)
display(classwise_cv_results.round(3))


,Model,Target class,Precision,Recall,F1,Support
0,Random Forest,Moderate,0.863,0.776,0.817,6759
1,Random Forest,Overcontroller,0.777,0.928,0.846,2263
2,Random Forest,Resilient,0.930,0.961,0.945,4930
3,Random Forest,Undercontroller,0.525,0.546,0.535,1820
4,Logistic Regression,Moderate,0.909,0.721,0.804,6759
5,Logistic Regression,Overcontroller,0.724,0.864,0.788,2263
6,Logistic Regression,Resilient,1.000,0.999,0.999,4930
7,Logistic Regression,Undercontroller,0.452,0.691,0.547,1820
8,KNN,Moderate,0.789,0.874,0.830,6759
9,KNN,Overcontroller,0.818,0.900,0.857,2263


## Selected champion and holdout evaluation

**Selection from the documented run (2026-10-02):** Logistic regression was selected despite random forest's marginally higher mean training-CV macro-F1 (0.785807 versus 0.784427; KNN: 0.744169). Random forest led by about 0.0014, whereas logistic regression was more stable across folds, faster, and easier to explain. For `Undercontroller`, out-of-fold F1 was 0.547 for logistic regression, 0.535 for random forest, and 0.351 for KNN. The best tested KNN configuration used `n_neighbors=11`, `weights="distance"`, and `p=2`. These results are descriptive evidence from that run, not a claim that the cleared notebook already contains fresh output.

The code below fixes the selected logistic-regression configuration and finds its **new** MLflow run ID after reproduction. It does not choose a new family based on regenerated scores or the holdout. If the selected configuration is missing or ambiguous, it stops for manual review before holdout evaluation.

The saved, complete pipeline receives raw `X_test` features and produces the final holdout predictions. Macro-F1 is primary; the class-wise report and confusion matrix show where performance varies. The current run ID must be deliberately copied into `model_loader.py` before running the app. The holdout results are not used for retrospective model selection.

In [14]:
# These constants record the prior manual champion decision.
# The old run ID is historical evidence and is not reused.
CHAMPION_MODEL_NAME = "Logistic Regression"
CHAMPION_C = 1.0984035416423363
CHAMPION_CLASS_WEIGHT = "balanced"

# Hyperopt/MLflow assign new run IDs on each execution.
# pd.to_numeric maps missing random-forest C values to NaN; they cannot match.
# np.isclose compares the floating-point parameter with a small tolerance.
current_c_values = pd.to_numeric(tuning_results["C"], errors="coerce")
matching_champion_runs = tuning_results.loc[
    tuning_results["Model"].eq(CHAMPION_MODEL_NAME)
    & np.isclose(current_c_values, CHAMPION_C, rtol=0, atol=1e-12)
    & tuning_results["class_weight"].eq(CHAMPION_CLASS_WEIGHT)
]
if len(matching_champion_runs) != 1:
    raise RuntimeError(
        "The manually selected champion configuration was not uniquely found in the current "
        "tuning run. Please review the current MLflow runs manually and "
        "reconfirm the champion before any holdout evaluation."
    )

# iloc[0] reads the sole match; only its technical run ID is updated.
CHAMPION_RUN_ID = matching_champion_runs.iloc[0]["MLflow run ID"]
# The f-string inserts the run ID into MLflow's artifact URI.
CHAMPION_MODEL_URI = f"runs:/{CHAMPION_RUN_ID}/model"

# Load the logged pipeline fitted on X_train; X_test has not been used for fitting.
# load_model() reconstructs imputation, scaling, encoding, and the classifier as one pipeline.
champion_pipeline = mlflow.sklearn.load_model(CHAMPION_MODEL_URI)
# predict() expects raw X_test features; manual preprocessing here would be incorrect and duplicate.
y_test_pred = champion_pipeline.predict(X_test)
class_labels = sorted(y.unique())

# Macro-F1 remains primary; accuracy and balanced accuracy add context.
holdout_metrics = pd.DataFrame(
    {
        "Metric": ["Macro-F1", "Accuracy", "Balanced Accuracy", "Test rows"],
        "Value": [
            f1_score(y_test, y_test_pred, average="macro"),
            accuracy_score(y_test, y_test_pred),
            balanced_accuracy_score(y_test, y_test_pred),
            len(y_test),
        ],
    }
)

print(f"Champion: {CHAMPION_MODEL_NAME}")
print(f"MLflow run ID: {CHAMPION_RUN_ID}")
display(holdout_metrics.round(3))

# The text report shows precision, recall, and F1 by class; it is not used to revisit model choice.
print("Class-wise holdout report:")
print(classification_report(y_test, y_test_pred, labels=class_labels, digits=3, zero_division=0))

# The confusion matrix counts actual classes by row and predicted classes by column.
confusion_matrix_table = pd.DataFrame(
    confusion_matrix(y_test, y_test_pred, labels=class_labels),
    index=pd.Index(class_labels, name="Actual"),
    columns=pd.Index(class_labels, name="Predicted"),
)
display(confusion_matrix_table)


Champion: Logistic Regression
MLflow run ID: 7d27288ce33e41d3b5490f8b04e9b07d


,Metric,Value
0,Macro-F1,0.784
1,Accuracy,0.824
2,Balanced Accuracy,0.819
3,Test rows,3943.000


Class-wise holdout report:
                 precision    recall  f1-score   support

       Moderate      0.909     0.720     0.804      1690
 Overcontroller      0.719     0.855     0.781       566
      Resilient      1.000     1.000     1.000      1232
Undercontroller      0.455     0.699     0.551       455

       accuracy                          0.824      3943
      macro avg      0.771     0.819     0.784      3943
   weighted avg      0.858     0.824     0.833      3943



Predicted,Moderate,Overcontroller,Resilient,Undercontroller
Actual,,,,
Moderate,1217,141,0,332
Overcontroller,33,484,0,49
Resilient,0,0,1232,0
Undercontroller,89,48,0,318


## Reproduction, interpretation, and app handoff

A controlled local reproduction completed all 60 Hyperopt trials and recovered the same selected logistic-regression configuration. It reported holdout macro-F1 of 0.784 on 3,943 rows. Class-wise F1 differed substantially: `Resilient` reached 1.000, while `Undercontroller` reached 0.551. Because the target labels were derived from questionnaire information in this course dataset, these scores do not establish independent clinical validity or reliability for other populations.

Run all cells from the project root with the cleaned dataset available. Review the regenerated CV table, class-wise comparison, MLflow runs, and holdout report against the documented selection. A new execution creates a new MLflow run ID; transfer the ID for the confirmed logistic-regression configuration to `model_loader.py` and test the app against the same local MLflow store. Reassess any unexpected result before using it in a report or presentation.